In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=None, stop_sequences=None, tools=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if temperature is not None:
        params["temperature"] = temperature

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if tools:
        params["tools"] = tools

    return client.messages.create(**params)


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

In [3]:
# Web search is a *server* tool: Anthropic runs the searches inside a single
# API request and hands Claude the results — there is no client-side execution
# loop and no input_schema to define. Claude decides when to search: it
# searches for current or changing information and answers directly from
# stable knowledge.
#
# `web_search_20260318` is the latest version. By default it runs with dynamic
# filtering: Claude writes code (via code execution) that filters the results
# before they reach the context window. Setting `allowed_callers: ["direct"]`
# opts into a plain direct search instead, which keeps the response structure
# simple and returns citations on the answer — ideal for studying the tool.
# `max_uses` caps the number of searches per request, useful because searches
# are billed per use ($10 per 1,000) on top of token costs.
web_search_schema = {
    "type": "web_search_20260318",
    "name": "web_search",
    "allowed_callers": ["direct"],
    "max_uses": 5,
}

In [4]:
# A question about current information, so Claude actually searches instead of
# answering from its training data.
messages = []
add_user_message(
    messages,
    "What is the latest stable version of Python, and when was it released?",
)
response = chat(messages, tools=[web_search_schema])

print(text_from_message(response))

The latest stable major version of Python is **Python 3.14**, which was 
released on October 7, 2025
. 
Python 3.14 is the latest stable release of the Python programming language, with a mix of changes to the language, the implementation, and the standard library, with the biggest changes including template string literals, deferred evaluation of annotations, and support for subinterpreters in the standard library.


Since the initial 3.14.0 release, there have been maintenance updates: 
Python 3.14.3 was released Feb. 3, 2026, Python 3.14.2 on Dec. 5, 2025, and Python 3.14.1 on Dec. 2, 2025
, with 3.14.3 being the most recent bugfix release as of early 2026.


In [5]:
# The response interleaves Claude's text with server tool blocks:
# `server_tool_use` (each query Claude ran) and `web_search_tool_result`
# (the results, or a single error object). Text drawn from results carries
# citations, and usage reports how many searches were billed.
for block in response.content:
    if block.type == "server_tool_use" and block.name == "web_search":
        print(f">>> Search: {block.input.get('query')}")
    elif block.type == "web_search_tool_result":
        if isinstance(block.content, list):
            for result in block.content:
                print(f"    - {result.title} ({result.url})")
        else:
            print(f"    Search error: {block.content.error_code}")

cited_sources = {
    citation.url: citation.title
    for block in response.content
    if block.type == "text" and block.citations
    for citation in block.citations
}
print("\nCited sources:")
for url, title in cited_sources.items():
    print(f"- {title}: {url}")

if response.usage.server_tool_use:
    print(f"\nSearches billed: {response.usage.server_tool_use.web_search_requests}")

>>> Search: latest stable version of Python release date
    - Status of Python versions (https://devguide.python.org/versions/)
    - Python | endoflife.date (https://endoflife.date/python)
    - Download Python | Python.org (https://www.python.org/downloads/)
    - Python documentation by version | Python.org (https://www.python.org/doc/versions/)
    - Python latest stable version? (2026) (https://solatatech.com/article/python-latest-stable-version)
    - The Latest Version of Python | phoenixNAP KB (https://phoenixnap.com/kb/latest-python-version)
    - Latest Python Version (2025) – What's New in Python 3.14? (https://www.liquidweb.com/blog/latest-python-version/)
    - Python Release Python 3.14.0 | Python.org (https://www.python.org/downloads/release/python-3140/)
    - Python Source Releases for Source release (https://www.python.org/downloads/source/)
    - What's new in Python 3.14 — Python 3.14.6 documentation (https://docs.python.org/3/whatsnew/3.14.html)

Cited sources:
- 